# Emotion head: better emotions from emotion2vec

Needs `colab_setup.ipynb` run first in this Colab session (same runtime), and the latest code pushed to GitHub.

Today step 3 uses emotion2vec's own 7-way output: 35-38% on Rasa Hindi/Tamil. This tries:
- removing each speaker's bias (no labels needed, only several lines from the same voice);
- a small head trained on RAVDESS embeddings.

Rasa is **evaluation only**: nothing is trained on it. No Groq used.

In [ ]:
# 1. Paths, latest code, and a check that setup ran in this session
import os, shutil, subprocess
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/nlp-project-clips")
REPO = Path("/content/nlp-project")
PY = str(REPO / ".venv/bin/python")
ENV = {**os.environ, "HF_HOME": str(DRIVE / "hf-cache"), "PYTHONPATH": str(REPO)}
assert Path(PY).exists(), "Run colab_setup.ipynb first in this runtime"
print(subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], capture_output=True, text=True).stdout.strip())
assert (REPO / "scripts/emotion_embed.py").exists(), "Push the latest code to GitHub first, then rerun this cell"
SAVED = DRIVE / "outputs" / "emotion"
SAVED.mkdir(parents=True, exist_ok=True)
print("ready")

In [ ]:
%%bash
# 2. Data: RAVDESS (~200 MB) and the Rasa Hindi + Tamil test sets (~3.5 GB), on Colab's disk (~5 min)
cd /content/nlp-project
bash scripts/fetch_ravdess.sh
HF_HOME=/content/drive/MyDrive/nlp-project-clips/hf-cache .venv/bin/python scripts/fetch_rasa.py --lang hi ta 2>&1 | grep -v Fetching

In [ ]:
# 3. emotion2vec embeddings on the GPU (~5-10 min), kept on Drive so a rerun skips this
cache = REPO / "data/cache"
cache.mkdir(parents=True, exist_ok=True)
for name in ["ravdess", "rasa_hi", "rasa_ta"]:
    f = SAVED / f"e2v_{name}.npz"
    if f.exists():
        shutil.copy(f, cache / f.name)
        print(f"{name}: from Drive")
        continue
    r = subprocess.run([PY, "scripts/emotion_embed.py", name], cwd=REPO, env=ENV, capture_output=True, text=True)
    print(r.stdout.strip() if r.returncode == 0 else "FAILED\n" + r.stderr[-2000:])
    if r.returncode == 0:
        shutil.copy(cache / f.name, f)

In [ ]:
# 4. Compare the methods on Rasa Hindi + Tamil (CPU, under a minute)
r = subprocess.run([PY, "scripts/train_emotion_head.py"], cwd=REPO, env=ENV, capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else "FAILED\n" + r.stderr[-3000:])